In [3]:
import sys
sys.path.append('..')

In [2]:
import logging
from pathlib import Path
import shutil
import pandas as pd
import os
from pyedflib import highlevel
import subprocess
from config.paths import PatientDir

In [13]:
old_root = Path('/data/home/webb/UNEEG_data_ultra/')
root = Path('/data/home/webb/original_UNEEG/')
data_dir = root / 'data'

In [10]:
# Make edf dirs a subfolder
for pdir in old_root.iterdir():
    pname = pdir.name
    new_pdir = data_dir / pname
    new_pdir.mkdir(exist_ok=True)
    # the pdir is renamed to edf_data in the correct folder
    pdir.rename(new_pdir / 'edf_data')

In [16]:
# Copy competition training data
src_dir = Path('/data/datasets/20250501_SUBQ_SeizurePredictionCompetition_2025final/TrainingData')
for i in [1, 2, 3]:
    src = src_dir / f'TrainingP{i}'
    dst = data_dir / f'competition-{i}' / src.name
    shutil.copytree(src, dst, copy_function=shutil.copy2)

In [18]:
# make seizure annotation folders
for pdir in data_dir.iterdir():
    anns_original_dir = pdir / 'seizure_annotations' / 'original'
    anns_original_dir.mkdir(exist_ok=True, parents=True)

In [19]:
# move competition data to a single edf folder
for i in [1, 2, 3]:
    pdir = data_dir / f'competition-{i}'
    train_dir = pdir / f'TrainingP{i}'
    test_dir = pdir / f'P{i}Testing'
    edf_dir = pdir/ 'edf_data'
    edf_dir.mkdir(exist_ok=True)

    # Move files from train and test directories
    for src in [train_dir, test_dir]:
        if src.exists():
            for file in src.iterdir():
                if file.is_file():
                    file.rename(edf_dir / file.name)

            # Remove the now empty directory
            try:
                src.rmdir()
            except OSError:
                print(f"Directory not empty or could not be removed: {src}")

In [29]:
# Try loading seizure starts
szr_starts_path = data_dir / 'competition-3' / 'seizure_annotations' / 'original' / 'seizure_starts.csv'
szr_starts = pd.read_csv(szr_starts_path, parse_dates=['start'])
szr_starts

,start
0,2022-01-29 03:53:11
1,2022-01-30 05:51:19
2,2022-02-13 04:57:23
3,2022-02-13 08:38:09
4,2022-02-27 01:22:07
...,...
58,2023-02-10 03:00:43
59,2023-02-20 06:17:29
60,2023-02-25 04:53:23
61,2023-03-13 02:37:01


In [8]:
# check out duplicated files
os.chdir(data_dir)

'/data/home/webb/original_UNEEG/data'

In [11]:
signals, signal_header, header = highlevel.read_edf("./U002-DE01-16/edf_data/U002-DE01-16_OUTPT_V6_000538_0004003_20000403_03_EEGdata.edf")

In [ ]:
# Delete duplicate files in each patient folder
all_deleted_files = []
pdirs = [PatientDir(p) for p in data_dir.iterdir()]
for pdir in pdirs:
    target_dir = pdir.edf_dir
    logging.info(f'Processing duplicates for {pdir.name}')
    dups = subprocess.run

In [37]:
# correct_competition_ptnt_sequence_file_sequence_number
import re
from config.paths import PATHS, Dataset
import pandas as pd
from pathlib import Path

# for pdir in PATHS.patient_dirs([Dataset.competition]):
pdir = PATHS.patient_dirs([Dataset.competition])[2]
print(pdir.name)
edfs = pd.DataFrame({'path': pdir.edf_dir.iterdir()})

def get_competition_sequence_number(path: Path) -> int:
    """
    Extracts the sequence number from filenames:
    - P1_287.edf -> 287
    - file_794_SQ1317.edf -> 1317
    """
    name = path.stem
    # Try finding SQXXXX first
    sq_match = re.search(r'SQ(\d+)', name)
    if sq_match: return int(sq_match.group(1))
    # Fallback to the last numeric part (e.g., P1_287 -> 287)
    num_match = re.findall(r'\d+', name)
    if num_match: return int(num_match[-1])
    raise ValueError(f"Couldn't extract sequence number from {name}")


edfs['seq'] = edfs['path'].apply(get_competition_sequence_number)
edfs.sort_values(by='seq', inplace=True)
edfs.reset_index(drop=True, inplace=True)

# Check where sequence and index are unequal
unequal_mask = edfs['seq'] != edfs.index
if any(unequal_mask):
    print(edfs[unequal_mask])

# Rename
edfs["filename"] = edfs.apply(
    lambda row: f"{row.path.parent.parent.name}_{row.name:04d}.edf",
    axis=1
)
edfs


competition-3
                                                   path   seq
220   /data/home/webb/original_UNEEG/datasets/compet...   221
221   /data/home/webb/original_UNEEG/datasets/compet...   223
222   /data/home/webb/original_UNEEG/datasets/compet...   225
223   /data/home/webb/original_UNEEG/datasets/compet...   227
224   /data/home/webb/original_UNEEG/datasets/compet...   228
...                                                 ...   ...
1883  /data/home/webb/original_UNEEG/datasets/compet...  1892
1884  /data/home/webb/original_UNEEG/datasets/compet...  1893
1885  /data/home/webb/original_UNEEG/datasets/compet...  1894
1886  /data/home/webb/original_UNEEG/datasets/compet...  1895
1887  /data/home/webb/original_UNEEG/datasets/compet...  1896

[1668 rows x 2 columns]


,path,seq,filename
0,/data/home/webb/original_UNEEG/datasets/compet...,0,competition-3_0000.edf
1,/data/home/webb/original_UNEEG/datasets/compet...,1,competition-3_0001.edf
2,/data/home/webb/original_UNEEG/datasets/compet...,2,competition-3_0002.edf
3,/data/home/webb/original_UNEEG/datasets/compet...,3,competition-3_0003.edf
4,/data/home/webb/original_UNEEG/datasets/compet...,4,competition-3_0004.edf
...,...,...,...
1883,/data/home/webb/original_UNEEG/datasets/compet...,1892,competition-3_1883.edf
1884,/data/home/webb/original_UNEEG/datasets/compet...,1893,competition-3_1884.edf
1885,/data/home/webb/original_UNEEG/datasets/compet...,1894,competition-3_1885.edf
1886,/data/home/webb/original_UNEEG/datasets/compet...,1895,competition-3_1886.edf


In [38]:
edfs[unequal_mask]

,path,seq,filename
220,/data/home/webb/original_UNEEG/datasets/compet...,221,competition-3_0220.edf
221,/data/home/webb/original_UNEEG/datasets/compet...,223,competition-3_0221.edf
222,/data/home/webb/original_UNEEG/datasets/compet...,225,competition-3_0222.edf
223,/data/home/webb/original_UNEEG/datasets/compet...,227,competition-3_0223.edf
224,/data/home/webb/original_UNEEG/datasets/compet...,228,competition-3_0224.edf
...,...,...,...
1883,/data/home/webb/original_UNEEG/datasets/compet...,1892,competition-3_1883.edf
1884,/data/home/webb/original_UNEEG/datasets/compet...,1893,competition-3_1884.edf
1885,/data/home/webb/original_UNEEG/datasets/compet...,1894,competition-3_1885.edf
1886,/data/home/webb/original_UNEEG/datasets/compet...,1895,competition-3_1886.edf


In [40]:
# Rename
for i, edf in edfs.iterrows():
    new_path = edf.path.parent / edf.filename
    if new_path != edf.path:
        print(new_path)
        edf.path.rename(new_path)

/data/home/webb/original_UNEEG/datasets/competition/competition-3/edf_data/competition-3_0000.edf
/data/home/webb/original_UNEEG/datasets/competition/competition-3/edf_data/competition-3_0001.edf
/data/home/webb/original_UNEEG/datasets/competition/competition-3/edf_data/competition-3_0002.edf
/data/home/webb/original_UNEEG/datasets/competition/competition-3/edf_data/competition-3_0003.edf
/data/home/webb/original_UNEEG/datasets/competition/competition-3/edf_data/competition-3_0004.edf
/data/home/webb/original_UNEEG/datasets/competition/competition-3/edf_data/competition-3_0005.edf
/data/home/webb/original_UNEEG/datasets/competition/competition-3/edf_data/competition-3_0006.edf
/data/home/webb/original_UNEEG/datasets/competition/competition-3/edf_data/competition-3_0007.edf
/data/home/webb/original_UNEEG/datasets/competition/competition-3/edf_data/competition-3_0008.edf
/data/home/webb/original_UNEEG/datasets/competition/competition-3/edf_data/competition-3_0009.edf
/data/home/webb/orig